In [ ]:
from paths import ROOT
import sys
!{sys.executable} -m pip install fastparquet

In [ ]:
"""
Convert all CSV files in two OECD ICIO directories to Parquet (zstd).

Why Parquet (not npz):
  - Rows and columns are labeled (country_sector codes) -- npz would drop them.
  - Mixed int64/float64 columns -- Parquet preserves per-column dtypes;
    npz would force a single dtype for the whole matrix.
  - Compression on columnar data with many repeated/sparse values is excellent.
"""

import time
from pathlib import Path
import pandas as pd


In [ ]:
def human(num_bytes: float) -> str:
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if num_bytes < 1024:
            return f"{num_bytes:7.2f} {unit}"
        num_bytes /= 1024
    return f"{num_bytes:7.2f} PB"


def convert_one(csv_path: Path) -> dict:
    """Convert a single CSV to Parquet next to it. Returns a stats dict."""
    parquet_path = csv_path.with_suffix(".parquet")
    stats = {
        "csv": csv_path,
        "parquet": parquet_path,
        "csv_bytes": csv_path.stat().st_size,
        "parquet_bytes": None,
        "rows": None,
        "cols": None,
        "elapsed_s": None,
        "status": None,
    }

    if parquet_path.exists() and not OVERWRITE:
        stats["status"] = "skipped (exists)"
        stats["parquet_bytes"] = parquet_path.stat().st_size
        if DELETE_CSV and csv_path.exists():
            # verify before deleting
            try:
                check = pd.read_parquet(parquet_path)
                if check.shape[1] + 1 >= 2:  # basic sanity
                    csv_path.unlink()
                    stats["status"] = "skipped (exists, csv deleted)"
            except Exception:
                pass
        return stats

    t0 = time.perf_counter()
    # index_col=0 -> the first column ("V1") holds the row labels (country_sector codes)
    df = pd.read_csv(csv_path, index_col=0)
    stats["rows"], stats["cols"] = df.shape

    if DRY_RUN:
        stats["status"] = "dry-run"
        stats["elapsed_s"] = time.perf_counter() - t0
        return stats

    # Write to a temp file then rename: avoids leaving a half-written .parquet
    # if the process is interrupted mid-write.
    tmp = parquet_path.with_suffix(".parquet.tmp")
    df.to_parquet(tmp, compression="gzip", engine="fastparquet", index=True)
    tmp.replace(parquet_path)

    stats["parquet_bytes"] = parquet_path.stat().st_size
    stats["elapsed_s"] = time.perf_counter() - t0
    stats["status"] = "ok"

    if DELETE_CSV:
        # Sanity-check before deleting: re-read the parquet and compare shape.
        check = pd.read_parquet(parquet_path)
        if check.shape == df.shape:
            csv_path.unlink()
            stats["status"] = "ok (csv deleted)"
        else:
            stats["status"] = "ok (csv KEPT: shape mismatch on verify)"

    return stats


def find_csvs(root: Path):
    """Recursively yield all .csv files under root (sorted for stable output)."""
    return sorted(root.rglob("*.csv"))


def main():
    grand_csv = 0
    grand_parquet = 0
    rows_log = []

    for root in ROOTS:
        if not root.exists():
            print(f"!! Missing root: {root}")
            continue

        csvs = find_csvs(root)
        print(f"\n=== {root}")
        print(f"    {len(csvs)} CSV file(s) found")

        for csv in csvs:
            try:
                s = convert_one(csv)
            except Exception as e:
                print(f"  FAIL  {csv.relative_to(root)}  ->  {type(e).__name__}: {e}")
                continue

            rel = csv.relative_to(root)
            ratio = (s["parquet_bytes"] / s["csv_bytes"]) if s["parquet_bytes"] else None
            shape_str = f"{s['rows']}x{s['cols']}" if s["rows"] else "-"
            t_str = f"{s['elapsed_s']:5.1f}s" if s["elapsed_s"] is not None else "    -"
            ratio_str = f"{ratio:5.1%}" if ratio is not None else "    -"
            print(
                f"  {s['status']:24s}  {human(s['csv_bytes'])} -> "
                f"{human(s['parquet_bytes']) if s['parquet_bytes'] else '       -   '}  "
                f"({ratio_str})  {shape_str:>12s}  {t_str}  {rel}"
            )
            grand_csv += s["csv_bytes"]
            grand_parquet += s["parquet_bytes"] or 0
            rows_log.append(s)

    print("\n=== Totals ===")
    print(f"  CSV     : {human(grand_csv)}")
    print(f"  Parquet : {human(grand_parquet)}")
    if grand_csv:
        print(f"  Ratio   : {grand_parquet / grand_csv:.1%}  "
              f"(saved {human(grand_csv - grand_parquet)})")




In [ ]:
ROOTS = [
    ROOT / "data/interim/IOT/OCDE ICIO",
    ROOT / "data/interim/IOT/OCDE ICIO aggregated",
]

COMPRESSION = "zstd"      # best size/speed compromise (snappy = faster but ~30% bigger)
DRY_RUN = False           # True = report what would happen, don't write
DELETE_CSV = False        # True = remove source CSV after a successful, verified write
OVERWRITE = False         # True = re-convert even if .parquet already exists


if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd

csv_path = str(ROOT / "data/interim/IOT/OCDE ICIO/1995-2000_SML/1995_SML.csv")
parquet_path = str(ROOT / "data/interim/IOT/OCDE ICIO/1995-2000_SML/1995_SML.parquet")

csv = pd.read_csv(csv_path, index_col=0)
pq = pd.read_parquet(parquet_path, engine="fastparquet")

assert csv.shape == pq.shape, f"shape mismatch: {csv.shape} vs {pq.shape}"
assert list(csv.index) == list(pq.index), "index mismatch"
assert list(csv.columns) == list(pq.columns), "columns mismatch"
assert csv.equals(pq), "values mismatch"

print("OK:", csv.shape)


In [ ]:
ROOTS = [
    ROOT / "data/interim/IOT/OCDE ICIO",
    ROOT / "data/interim/IOT/OCDE ICIO aggregated",
]

COMPRESSION = "zstd"      # best size/speed compromise (snappy = faster but ~30% bigger)
DRY_RUN = False           # True = report what would happen, don't write
DELETE_CSV = True        # True = remove source CSV after a successful, verified write
OVERWRITE = False         # True = re-convert even if .parquet already exists


if __name__ == "__main__":
    main()

In [ ]:
from pathlib import Path
import pandas as pd

ROOTS = [
    ROOT / "data/interim/IOT/OCDE ICIO",
    ROOT / "data/interim/IOT/OCDE ICIO aggregated",
]
DRY_RUN = False   # set False to actually delete
ENGINE = "fastparquet"   # or "pyarrow" depending on what you used

deleted = kept = missing_pq = mismatch = 0

for root in ROOTS:
    for csv in sorted(root.rglob("*.csv")):
        pq = csv.with_suffix(".parquet")
        if not pq.exists():
            print(f"  no parquet  : {csv.relative_to(root)}")
            missing_pq += 1
            continue
        try:
            df_csv = pd.read_csv(csv, index_col=0)
            df_pq = pd.read_parquet(pq, engine=ENGINE)
        except Exception as e:
            print(f"  read error  : {csv.relative_to(root)}  -> {e}")
            kept += 1
            continue

        ok = (
            df_csv.shape == df_pq.shape
            and list(df_csv.index) == list(df_pq.index)
            and list(df_csv.columns) == list(df_pq.columns)
        )
        if not ok:
            print(f"  MISMATCH    : {csv.relative_to(root)}  csv={df_csv.shape} pq={df_pq.shape}")
            mismatch += 1
            kept += 1
            continue

        if DRY_RUN:
            print(f"  would delete: {csv.relative_to(root)}")
        else:
            csv.unlink()
            print(f"  deleted     : {csv.relative_to(root)}")
        deleted += 1

print(f"\n{'Would delete' if DRY_RUN else 'Deleted'}: {deleted}   kept: {kept}   "
      f"no-parquet: {missing_pq}   shape-mismatch: {mismatch}")
